
# Speech-to-Text & Text-to-Speech (Hugging Face) — Colab-ready Notebook

This notebook is a teaching resource for a class on **Speech-to-Text (ASR)** and **Text-to-Speech (TTS)**. It uses **Hugging Face models** and other open-source tools, with examples for **English** and **Bengali (Bangla)**.

Features included:
- Install & setup (Colab-friendly)
- ASR: Whisper (multilingual) + Bengali Wav2Vec2 models
- TTS: Facebook MMS (English & Bengali) + community Bengali TTS
- Voice cloning / zero-shot cloning using Coqui XTTS (local in Colab)
- Saving and playing audio in the notebook

Notes:
- First run requires downloading models (internet). After that, caching enables offline use.
- Stable GPU (Colab GPU) is recommended for faster model performance but not required for all examples.

---

## 0) Install required libraries (run in Colab)

In [ ]:
# Run these in Colab (uncomment to execute)
# !pip install -q  transformers==4.39.3 datasets==2.18.0 soundfile==0.12.1 librosa==0.10.2
!pip install -q "huggingface_hub>=0.14.1"
# !pip install -q transformers[speech]
!pip install -q git+https://github.com/coqui-ai/TTS.git@main  # for XTTS voice-cloning
!pip install -q soundfile
!pip install -q ipywidgets
# !pip install -q torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


text -> speech (tts) | speech -> text (stt) | chatbot (text) -> text

openai realtime api -> speech - speech

# If you run into CUDA/torch wheel issues on Colab, follow colab recommended torch install.

## 1) Utilities: load audio, play, save

In [ ]:
import torchaudio
import soundfile as sf
from IPython.display import Audio, display
import requests
from io import BytesIO
import os


In [ ]:
# Helper: download audio from URL
def load_audio_from_url(url, target_sr=16000):
    resp = requests.get(url)
    audio_bytes = BytesIO(resp.content)
    data, sr = sf.read(audio_bytes)
    # convert stereo to mono if needed
    if len(data.shape) > 1:
        data = data.mean(axis=1)
    # resample if needed
    if sr != target_sr:
        data = torchaudio.functional.resample(torch.tensor(data).unsqueeze(0), sr, target_sr).squeeze(0).numpy()
        sr = target_sr
    return data, sr

# Helper: save numpy audio to file
def save_audio_np(waveform, sr, outpath):
    sf.write(outpath, waveform, sr)

# Play audio in notebook
def play_audio_file(path, autoplay=False):
    display(Audio(path, autoplay=autoplay))

In [ ]:
print('utils loaded')


utils loaded


https://deepgram.com/ ->STT

In [ ]:
gpu -> model host-> latency | performance

## 2) ASR: English — Whisper (Hugging Face `transformers` pipeline)


We'll use the `pipeline('automatic-speech-recognition')` with an OpenAI Whisper model. Whisper is robust & multilingual.


In [ ]:
from transformers import pipeline

print('Loading Whisper ASR pipeline (this will download model weights)')
asr_whisper = pipeline('automatic-speech-recognition', model='openai/whisper-small')


print('Transcribing...')
res = asr_whisper('/content/LJ001-0002.wav')
print('Transcription:', res['text'])
# play_audio_file('en_example.wav') # Uncomment if 'en_example.wav' exists

Loading Whisper ASR pipeline (this will download model weights)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json: 0.00B [00:00, ?B/s]

Transcribing...


Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


Transcription:  in being comparatively modern.


In [ ]:
# call recording -> audio split -> model -> Transcription

# stt (model)-> opensource

##finetuning

In [ ]:
play_audio_file('LJ001-0002.wav')

In [ ]:
print('Transcribing...')
res = asr_whisper('/content/LJ001-0006.wav')
print('Transcription:', res['text'])
# play_audio_file('en_example.wav') # Uncomment if 'en_example.wav' exists

Transcribing...
Transcription:  And it is worth mentioning in passing that as an example of fine typography


In [ ]:
play_audio_file('LJ001-0006.wav')

## 3) ASR: Bengali — Wav2Vec2 models


Many community models fine-tuned for Bengali exist (wav2vec2 / XLSR). We provide two options: a small demo model and a stronger XLSR fine-tuned checkpoint.

- `arijitx/wav2vec2-xls-r-300m-bengali` (fine-tuned XLSR)
- `ai4bharat/indicwav2vec_v1_bengali` (another community model)

We'll use the pipeline API for simplicity.


In [ ]:
print('Loading Bengali ASR (wav2vec2) — may take a moment')
asr_bengali = pipeline('automatic-speech-recognition', model='arijitx/wav2vec2-xls-r-300m-bengali')

# Example Bengali short audio (upload your own or replace URL)
# NOTE: Provide your own Bengali audio in Colab by uploading or using a URL
# Here's a placeholder: you should replace with a proper Bengali audio file.

# If you have a file 'bn_example.wav' in Colab, run:
res_bn = asr_bengali('/content/train_barishal (1).wav')
print('Bengali transcription:', res_bn['text'])

Loading Bengali ASR (wav2vec2) — may take a moment


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of the model checkpoint at arijitx/wav2vec2-xls-r-300m-bengali were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at arijitx/wav2vec2-xls-r-300m-bengali and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably T

...

special_tokens_map.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Could not load the `decoder` for arijitx/wav2vec2-xls-r-300m-bengali. Defaulting to raw CTC. Error: No module named 'kenlm'
Try to install `kenlm`: `pip install kenlm
Try to install `pyctcdecode`: `pip install pyctcdecode


Bengali transcription: ময়লাইকুম আমার নাম হাসিবুরযা ও বিপরম্যান অফটফটয়ার ইনডনারিং ফাটটেলটয়ার ্েসমাম্বল ফোর্টি মি ডেপর্ি ইউ্া ন্যাশাল ইউনুভর্সিিতে পরশোনা করতৃরি আর আমানিংটযার হয়েলো বরিশালে।


In [ ]:
print('Loading Bengali ASR (wav2vec2) — may take a moment')
asr_bengali = pipeline('automatic-speech-recognition', model='ai4bharat/indicwav2vec_v1_bengali')

# Example Bengali short audio (upload your own or replace URL)
# NOTE: Provide your own Bengali audio in Colab by uploading or using a URL
# Here's a placeholder: you should replace with a proper Bengali audio file.

# If you have a file 'bn_example.wav' in Colab, run:
res_bn = asr_bengali('/content/train_barishal (1).wav')
print('Bengali transcription:', res_bn['text'])

Loading Bengali ASR (wav2vec2) — may take a moment


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of the model checkpoint at ai4bharat/indicwav2vec_v1_bengali were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at ai4bharat/indicwav2vec_v1_bengali and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/940 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

Bengali transcription: ময়ালাইকুম আমার নাম হাসিবুর রহমান সুকও ডিপার্মেন্ট অব সফটওয়ার ইঞ্জিনিয়ারিং ফাটফরি মির বযাসনাম্বার ফোর্ডটি আমি ডেফুডিন ইন্টারন্যাশনাল ইউনিভার্সিটিতে পড়াশোনা করতেছি আর আমার হিংটন হচ্ছে ওল বরিশালে


## 4) TTS: English & Bengali


In [ ]:
# Install the latest version of transformers and accelerate
# !pip install -q --upgrade transformers accelerate scipy soundfile
import torch

from transformers import VitsModel, AutoTokenizer, set_seed
import scipy.io.wavfile as wavfile
from IPython.display import Audio

print("Loading MMS-TTS English model...")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")
model = VitsModel.from_pretrained("facebook/mms-tts-eng")

text = "Hello, students! This is a demo of English TTS using Facebook MMS."
inputs = tokenizer(text, return_tensors="pt")

# Set seed for deterministic output
set_seed(42)

with torch.no_grad():
    outputs = model(**inputs).waveform

waveform = outputs[0].cpu().numpy()
sr = model.config.sampling_rate

wavfile.write("mms_tts_eng.wav", rate=sr, data=waveform)
print("Saved as mms_tts_eng.wav — sample below:")

Audio("mms_tts_eng.wav", rate=sr)


Loading MMS-TTS English model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/mms-tts-eng were not used when initializing VitsModel: ['flow.flows.0.wavenet.in_layers.0.weight_g', 'flow.flows.0.wavenet.in_layers.0.weight_v', 'flow.flows.0.wavenet.in_layers.1.weight_g', 'flow.flows.0.wavenet.in_layers.1.weight_v', 'flow.flows.0.wavenet.in_layers.2.weight_g', 'flow.flows.0.wavenet.in_layers.2.weight_v', 'flow.flows.0.wavenet.in_layers.3.weight_g', 'flow.flows.0.wavenet.in_layers.3.weight_v', 'flow.flows.0.wavenet.res_skip_layers.0.weight_g', 'flow.flows.0.wavenet.res_skip_layers.0.weight_v', 'flow.flows.0.wavenet.res_skip_layers.1.weight_g', 'flow.flows.0.wavenet.res_skip_layers.1.weight_v', 'flow.flows.0.wavenet.res_skip_layers.2.weight_g', 'flow.flows.0.wavenet.res_skip_layers.2.weight_v', 'flow.flows.0.wavenet.res_skip_layers.3.weight_g', 'flow.flows.0.wavenet.res_skip_layers.3.weight_v', 'flow.flows.1.wavenet.in_layers.0.weight_g', 'flow.flows.1.wavenet.in_layers.0.weight_v', 'flow.flows.1.wavenet.in_layers.1.wei

Saved as mms_tts_eng.wav — sample below:


In [ ]:
# youtubevideo -> link -> audio -> transcription -> model(transcription)-> Refine .... -> summary

In [ ]:
assignment - voice to voice chatbot (stt , tts)
Exam week (Virtual Try on)

In [ ]:
# speech to speech chatbot
# speech -> stt -> text -> chatbot ->rag (agent) ->text -> tts -> speech

## 7) Tips & Colab pointers for the class

- **Use GPU runtime** in Colab for faster model downloads and generation (Runtime > Change runtime type > GPU).
- **Model caching**: HF models will be cached in `~/.cache/huggingface` on first use.
- **Rate limits**: For large class, students should run at different times or use small models to avoid HF rate limits.
- **Audio formats**: Ensure input audio is 16 kHz mono for many ASR models (wav). Use `torchaudio` or `librosa` for resampling.
- **Safety & Licenses**: Check model license (e.g., Coqui models may be non-commercial).

---

# Exercises for students
# 1. Compare Whisper and wav2vec2 on a short Bengali-English mixed audio file.
# 2. Fine-tune a small wav2vec2 model on a tiny Bengali dataset (use the `datasets` library).
# 3. Try zero-shot voice cloning with coqui XTTS: record a 6-second clip and synthesize a paragraph.

# End of notebook